# LSTM Delta Learning for Hydrogen Reaction Surrogate Model

## Experiment Overview
- **Goal:** Predict $\Delta y = y(t+h) - y(t)$ instead of absolute $y(t+h)$
- **Hypothesis:** Delta learning forces LSTM to learn derivatives, preventing identity mapping during slow phases
- **Dataset:** Basalt 25°C (14 state variables + pH)
- **Architecture:** Stacked LSTM (128 → 64)

## Key Design Decisions
1. **Uniform timesteps** - ODE solved on fixed grid (dt ≈ 0.008 days) for LSTM compatibility
2. **pH as input feature** - Helps model capture pH-dependent sulfide speciation
3. **Horizon = 30** - Based on W02 findings

---

## 1. Environment Setup

In [ ]:
# Check if running in Colab
import sys
IN_COLAB = 'google.colab' in sys.modules
print(f"Running in Colab: {IN_COLAB}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted.")

In [ ]:
# Install dependencies (if needed)
!pip install -q scipy scikit-learn matplotlib

In [ ]:
# Core imports
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
class Config:
    """Experiment configuration."""
    
    # === Data Generation ===
    T_START = 0.0
    T_END = 20.0
    N_POINTS = 2500  # Uniform grid: dt ≈ 0.008 days
    
    # === Train/Test Split ===
    TRAIN_SIZE = 2000
    TEST_SIZE = 500
    
    # === LSTM Architecture ===
    SEQ_LEN = 100       # Input sequence length
    HIDDEN_1 = 128      # First LSTM hidden size
    HIDDEN_2 = 64       # Second LSTM hidden size
    N_STATES = 14       # State variables from ODE
    N_FEATURES = 15     # States + pH
    
    # === Delta Learning ===
    PRED_HORIZON = 30   # Steps ahead to predict delta
    
    # === Training ===
    EPOCHS = 300
    BATCH_SIZE = 32
    LEARNING_RATE = 5e-4
    PATIENCE = 30       # Early stopping patience
    
    # === Forecast ===
    FORECAST_STEPS = 150
    
    # === Preprocessing ===
    # Columns to log-transform (small/zero-prone values)
    LOG_COLS = [3, 7, 9, 12, 13]  # nH2S_g, FeS, Acetate, Lag, Fe_pool
    
    # === Paths ===
    if IN_COLAB:
        # Update this path to your Google Drive location
        BASE_DIR = '/content/drive/MyDrive/thesis_data/Basalt_25C'
    else:
        BASE_DIR = r'd:\chemical_thesis_repo\2026-W02_Lstm_development\code\matlab\Basalt\25C'

config = Config()
print(f"Data directory: {config.BASE_DIR}")
print(f"Uniform grid: {config.N_POINTS} points over {config.T_END} days")
print(f"dt = {config.T_END / config.N_POINTS:.4f} days")

## 3. Upload Data Files (Colab Only)

If not using Google Drive, upload these files:
- `best_fit_params_Basalt_25C.mat`
- `Muller_2024_H2_Basalt_at_25C.txt`

In [ ]:
# Option A: Upload files directly (uncomment if not using Drive)
# from google.colab import files
# uploaded = files.upload()
# config.BASE_DIR = '/content'  # Files uploaded to /content

# Option B: Check if files exist in Drive path
mat_file = os.path.join(config.BASE_DIR, 'best_fit_params_Basalt_25C.mat')
txt_file = os.path.join(config.BASE_DIR, 'Muller_2024_H2_Basalt_at_25C.txt')

print(f"Checking files...")
print(f"  .mat file exists: {os.path.exists(mat_file)}")
print(f"  .txt file exists: {os.path.exists(txt_file)}")

if not os.path.exists(mat_file) or not os.path.exists(txt_file):
    print("\n⚠️ Files not found! Please either:")
    print("  1. Update config.BASE_DIR to your Google Drive path")
    print("  2. Uncomment the upload section above and upload files")

## 4. ODE Model (Two-Phase Anaerobic Model)

This is the Python implementation of the MATLAB v4 model with:
- 14 state variables
- Henry's Law gas-liquid transfer
- pH-dependent sulfide speciation
- Three microbial pathways

In [ ]:
def model_mixed(t, y, p, env):
    """
    Two-phase anaerobic model (v4) - Python implementation.
    
    State vector y (14 variables):
        [0] nH2_g    - H2 gas (mmol)
        [1] nCO2_g   - CO2 gas (mmol)
        [2] nCH4_g   - CH4 gas (mmol)
        [3] nH2S_g   - H2S gas (mmol)
        [4] H2_aq    - Dissolved H2 (mmol/L)
        [5] CO2_aq   - Dissolved CO2 (mmol/L)
        [6] SO4      - Sulfate (mmol/L)
        [7] FeS      - Iron sulfide precipitate (mmol/L)
        [8] X        - Biomass (mmol/L)
        [9] Acetate  - Acetate (mmol/L)
        [10] HCO3    - Bicarbonate (mmol/L, constant)
        [11] S_tot   - Total dissolved sulfide (mmol/L)
        [12] Lag     - Lag phase variable (0-1)
        [13] Fe_pool - Dissolved Fe(II) pool (mmol/L)
    """
    # Unpack environment
    Vg, Vl, T, Rgas = env['Vg'], env['Vl'], env['T'], env['Rgas']
    Hcp_H2 = env['Hcp_H2_eff']
    Hcp_CO2 = env['Hcp_CO2_eff']
    Hcp_H2S = env['Hcp_H2S_eff']
    pKa = env['pKa_H2S']
    pH = env['pH_fun'](t)
    
    # Guard against negative values
    y = np.maximum(y, 1e-12)
    Fe_pool = max(y[13], 0)
    
    # Unpack state
    nH2_g, nCO2_g, nCH4_g, nH2S_g = y[0], y[1], y[2], y[3]
    H2_aq, CO2_aq, SO4, FeS = y[4], y[5], y[6], y[7]
    X, Ac, HCO3, S_tot, Lag = y[8], y[9], y[10], y[11], y[12]
    
    # Unpack parameters (28 total)
    k_m, k_s, k_a = p[0], p[1], p[2]
    Y_m, Y_s, Y_a = p[3], p[4], p[5]
    KI_m, KI_s, KI_a = p[6], p[7], p[8]
    k_prec, HS_sat = p[9], p[10]
    H2_th, DG_th = p[11], p[12]
    K_H2, K_SO4, K_CO2 = p[13], p[14], p[15]
    kla_H2, kla_CO2, kla_H2S = p[16], p[17], p[18]
    b, t_lag, w_lag = p[19], p[20], p[21]
    k_diss_gyp, beta_SO4_m = p[22], p[23]
    
    # === Partial pressures (atm) ===
    pH2 = (nH2_g / 1000) * Rgas * T / Vg
    pCO2 = (nCO2_g / 1000) * Rgas * T / Vg
    pH2S = (nH2S_g / 1000) * Rgas * T / Vg
    
    # === Henry equilibrium concentrations ===
    Ceq_H2 = Hcp_H2 * pH2
    Ceq_CO2 = Hcp_CO2 * pCO2
    Ceq_H2S = Hcp_H2S * pH2S
    
    # === Gas-liquid transfer fluxes ===
    J_H2 = kla_H2 * (Ceq_H2 - H2_aq)
    J_CO2 = kla_CO2 * (Ceq_CO2 - CO2_aq)
    
    # === Sulfide speciation (pH-dependent) ===
    frac_HS = 1 / (1 + 10**(pKa - pH))
    HS_aq = S_tot * frac_HS
    H2S_aq = S_tot * (1 - frac_HS)
    Jout_H2S = kla_H2S * (H2S_aq - Ceq_H2S)
    
    # === Inhibition & activation factors ===
    f_inh_m = KI_m / (KI_m + HS_aq)
    f_inh_s = KI_s / (KI_s + HS_aq)
    f_inh_a = KI_a / (KI_a + HS_aq)
    f_H2 = H2_aq / (H2_aq + H2_th)
    f_lag = 1 / (1 + np.exp((t_lag - t) / max(w_lag, 1e-3)))
    f_act = f_H2 * f_lag
    
    # === Monod kinetics ===
    mH2 = H2_aq / (K_H2 + H2_aq)
    mSO4 = SO4 / (K_SO4 + SO4)
    mCO2 = CO2_aq / (K_CO2 + CO2_aq)
    
    # === Thermodynamic gates ===
    RT = 8.314e-3 * T
    Q_a = max(Ac, 1e-12) / (max(H2_aq, 1e-12)**4 * max(CO2_aq, 1e-12)**2)
    fT_s = 1 / (1 + np.exp((-152 + RT * np.log(1) - DG_th) / RT))
    fT_m = 1 / (1 + np.exp((-130 - DG_th) / RT))
    fT_a = 1 / (1 + np.exp((-95 + RT * np.log(Q_a) - DG_th) / RT))
    f_comp_m = 1 / (1 + beta_SO4_m * SO4)
    
    # === Reaction rates ===
    r_meth = k_m * X * mH2 * mCO2 * f_inh_m * f_act * fT_m * f_comp_m
    r_sulf = k_s * X * mH2 * mSO4 * f_inh_s * f_act * fT_s
    r_aceto = k_a * X * mH2 * (mCO2**2) * f_inh_a * f_act * fT_a
    r_prec = min(k_prec * max(0, HS_aq - HS_sat), Fe_pool)
    r_diss_gyp = k_diss_gyp * max(0, env['SO4_sat_gyp'] - SO4)
    
    # === Derivatives ===
    dy = np.zeros(14)
    dy[0] = -J_H2 * Vl                                    # nH2_g
    dy[1] = -J_CO2 * Vl                                   # nCO2_g
    dy[2] = r_meth * Vl                                   # nCH4_g
    dy[3] = Jout_H2S * Vl                                 # nH2S_g
    dy[4] = J_H2 - 4*(r_meth + r_sulf + r_aceto)          # H2_aq
    dy[5] = J_CO2 - r_meth - 2*r_aceto                    # CO2_aq
    dy[6] = -r_sulf + r_diss_gyp                          # SO4
    dy[7] = r_prec                                        # FeS
    dy[8] = Y_m*r_meth + Y_s*r_sulf + Y_a*r_aceto - b*X   # X (biomass)
    dy[9] = r_aceto                                       # Acetate
    dy[10] = 0.0                                          # HCO3 (constant)
    dy[11] = r_sulf - r_prec - Jout_H2S                   # S_tot
    dy[12] = (f_lag - Lag) / max(w_lag, 1e-3)             # Lag
    dy[13] = -r_prec                                      # Fe_pool
    
    return dy

## 5. Data Loading & Generation

In [ ]:
import scipy.io as sio
import pandas as pd

def load_and_generate_data(config):
    """
    Load MATLAB fitted parameters and generate ODE data on uniform grid.
    
    Returns:
        t_eval: Time array (uniform grid)
        data_full: (N, 15) array - 14 states + pH
        env: Environment dictionary for ODE
        p_fit: Fitted parameter vector
    """
    print("="*60)
    print("[1/6] Loading Data & Generating ODE Solution")
    print("="*60)
    
    # Load .mat file with fitted parameters
    mat_file = os.path.join(config.BASE_DIR, 'best_fit_params_Basalt_25C.mat')
    print(f"Loading: {mat_file}")
    mat = sio.loadmat(mat_file, squeeze_me=True, struct_as_record=False)
    p_fit = mat['p_fit']
    env_struct = mat['env']
    
    # Load experimental data for pH interpolation
    txt_file = os.path.join(config.BASE_DIR, 'Muller_2024_H2_Basalt_at_25C.txt')
    print(f"Loading: {txt_file}")
    df = pd.read_csv(txt_file, sep=r'\s+', comment='%', header=None, encoding='latin1')
    t_exp = df.values[:, 0]
    pH_exp = df.values[:, 5]
    
    # Create pH interpolation function
    pH_fun = interp1d(t_exp, pH_exp, kind='linear', fill_value='extrapolate')
    
    # Build environment dictionary
    env = {
        'Vg': float(env_struct.Vg),
        'Vl': float(env_struct.Vl),
        'T': float(env_struct.T),
        'Rgas': float(env_struct.Rgas),
        'Hcp_H2_eff': float(env_struct.Hcp_H2_eff),
        'Hcp_CO2_eff': float(env_struct.Hcp_CO2_eff),
        'Hcp_H2S_eff': float(env_struct.Hcp_H2S_eff),
        'pKa_H2S': float(env_struct.pKa_H2S),
        'SO4_sat_gyp': float(env_struct.SO4_sat_gyp),
        'pH_fun': pH_fun
    }
    
    print(f"\nEnvironment parameters:")
    print(f"  Vg = {env['Vg']:.3f} L, Vl = {env['Vl']:.4f} L")
    print(f"  T = {env['T']:.2f} K ({env['T']-273.15:.1f} °C)")
    print(f"  Hcp_H2 = {env['Hcp_H2_eff']:.2f} mmol/L/atm")
    
    # Initial conditions from experimental data
    nH2_g_0 = df.values[0, 1] / 1000.0  # µmol -> mmol
    nCO2_g_0 = df.values[0, 2] / 1000.0
    nCH4_g_0 = df.values[0, 3] / 1000.0
    nH2S_g_0 = df.values[0, 4] / 1000.0
    SO4_0 = df.values[0, 6]
    
    # Calculate initial aqueous equilibrium
    pH2_0 = (nH2_g_0 / 1000) * env['Rgas'] * env['T'] / env['Vg']
    pCO2_0 = (nCO2_g_0 / 1000) * env['Rgas'] * env['T'] / env['Vg']
    H2_aq0 = env['Hcp_H2_eff'] * pH2_0
    CO2_aq0 = env['Hcp_CO2_eff'] * pCO2_0
    
    y0 = np.array([
        nH2_g_0,    # [0] nH2_g
        nCO2_g_0,   # [1] nCO2_g
        nCH4_g_0,   # [2] nCH4_g
        nH2S_g_0,   # [3] nH2S_g
        H2_aq0,     # [4] H2_aq
        CO2_aq0,    # [5] CO2_aq
        SO4_0,      # [6] SO4
        0.01,       # [7] FeS
        0.01,       # [8] X (biomass)
        0.0,        # [9] Acetate
        0.0,        # [10] HCO3
        1.0,        # [11] S_tot (seed)
        0.0,        # [12] Lag
        0.10        # [13] Fe_pool
    ])
    
    print(f"\nInitial conditions:")
    print(f"  nH2_g = {y0[0]:.3f} mmol")
    print(f"  SO4 = {y0[6]:.2f} mmol/L")
    
    # Generate ODE solution on UNIFORM grid
    print(f"\nSolving ODE on uniform grid...")
    print(f"  Time range: [{config.T_START}, {config.T_END}] days")
    print(f"  Grid points: {config.N_POINTS}")
    
    t_eval = np.linspace(config.T_START, config.T_END, config.N_POINTS)
    
    sol = solve_ivp(
        lambda t, y: model_mixed(t, y, p_fit, env),
        [config.T_START, config.T_END],
        y0,
        t_eval=t_eval,
        method='BDF',
        rtol=1e-8,
        atol=1e-10
    )
    
    if not sol.success:
        raise RuntimeError(f"ODE solver failed: {sol.message}")
    
    data_states = sol.y.T  # (N, 14)
    
    # Augment with pH as 15th feature
    pH_vals = pH_fun(t_eval).reshape(-1, 1)
    data_full = np.hstack([data_states, pH_vals])  # (N, 15)
    
    print(f"\n✓ Data generated: {data_full.shape} (14 states + pH)")
    
    return t_eval, data_full, env, p_fit

In [ ]:
# Generate data
t_eval, data_full, env, p_fit = load_and_generate_data(config)

## 6. Visualize Generated Data

In [ ]:
# Plot key state variables
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

state_names = ['nH2_g (mmol)', 'nCO2_g (mmol)', 'nCH4_g (mmol)', 
               'nH2S_g (mmol)', 'SO4 (mmol/L)', 'pH']
state_indices = [0, 1, 2, 3, 6, 14]

for ax, name, idx in zip(axes.flat, state_names, state_indices):
    ax.plot(t_eval, data_full[:, idx], 'b-', linewidth=1.5)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel(name)
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
    
    # Mark train/test split
    split_time = t_eval[config.TRAIN_SIZE]
    ax.axvline(split_time, color='r', linestyle='--', alpha=0.5, label='Train/Test split')

plt.suptitle('Generated ODE Data (Basalt 25°C)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Preprocessing for Delta Learning

In [ ]:
def preprocess_delta(data, config):
    """
    Preprocess data for delta learning:
    1. Log-transform selected columns (small values)
    2. Normalize features
    3. Create delta targets: Δy = y(t+h) - y(t)
    
    Returns:
        all_feat_norm: Normalized full dataset
        X_train: Training sequences (N, SEQ_LEN, 15)
        Y_train: Delta targets (N, 14)
        feat_scaler: Feature scaler (for inverse transform)
        delta_scaler: Delta scaler (for inverse transform)
    """
    print("\n" + "="*60)
    print("[2/6] Preprocessing for Delta Learning")
    print("="*60)
    
    data_proc = data.copy()
    
    # Log transform selected columns (only states, not pH)
    log_indices = [c for c in config.LOG_COLS if c < config.N_STATES]
    print(f"Log-transforming columns: {log_indices}")
    data_proc[:, log_indices] = np.log1p(data_proc[:, log_indices])
    
    # Split train data
    train_data = data_proc[:config.TRAIN_SIZE]
    
    # Fit feature scaler on training data
    feat_scaler = StandardScaler()
    train_feat_norm = feat_scaler.fit_transform(train_data)
    all_feat_norm = feat_scaler.transform(data_proc)
    
    print(f"Feature normalization: mean≈0, std≈1")
    
    # Create sequences and delta targets
    horizon = config.PRED_HORIZON
    X_list, Y_delta_list = [], []
    
    for i in range(config.SEQ_LEN, config.TRAIN_SIZE - horizon):
        # Input: sequence of length SEQ_LEN
        seq = train_feat_norm[i - config.SEQ_LEN : i]
        X_list.append(seq)
        
        # Target: delta in NORMALIZED space
        current_state = train_feat_norm[i - 1, :config.N_STATES]
        future_state = train_feat_norm[i + horizon - 1, :config.N_STATES]
        delta = future_state - current_state
        Y_delta_list.append(delta)
    
    X_train = np.array(X_list)
    Y_train = np.array(Y_delta_list)
    
    # Normalize deltas
    delta_scaler = StandardScaler()
    Y_train_norm = delta_scaler.fit_transform(Y_train)
    
    print(f"\nDataset shapes:")
    print(f"  X_train: {X_train.shape} (samples, seq_len, features)")
    print(f"  Y_train: {Y_train_norm.shape} (samples, delta_states)")
    print(f"  Horizon: {horizon} steps")
    
    return all_feat_norm, X_train, Y_train_norm, feat_scaler, delta_scaler

In [ ]:
# Preprocess data
all_feat_norm, X_train, Y_train, feat_scaler, delta_scaler = preprocess_delta(data_full, config)

## 8. LSTM Model Definition

In [ ]:
class StackedLSTM(nn.Module):
    """
    Stacked LSTM for delta prediction.
    
    Architecture:
        Input (15) -> LSTM(128) -> LSTM(64) -> Linear(14)
    """
    def __init__(self, input_size, hidden1, hidden2, output_size, dropout=0.1):
        super().__init__()
        self.lstm1 = nn.LSTM(input_size, hidden1, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)
        self.lstm2 = nn.LSTM(hidden1, hidden2, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden2, output_size)
        
    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.dropout1(out)
        out, _ = self.lstm2(out)
        out = self.dropout2(out)
        return self.fc(out[:, -1, :])  # Last timestep output

# Create model
model = StackedLSTM(
    input_size=config.N_FEATURES,
    hidden1=config.HIDDEN_1,
    hidden2=config.HIDDEN_2,
    output_size=config.N_STATES
).to(device)

print(f"Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 9. Training

In [ ]:
def train_delta_model(model, X, Y, config, device):
    """
    Train LSTM with early stopping and learning rate scheduling.
    """
    print("\n" + "="*60)
    print(f"[3/6] Training Delta Model ({config.EPOCHS} max epochs)")
    print("="*60)
    
    # Create DataLoader
    dataset = TensorDataset(torch.FloatTensor(X), torch.FloatTensor(Y))
    loader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=True)
    
    # Optimizer and scheduler
    optimizer = torch.optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=20, factor=0.5, verbose=True
    )
    criterion = nn.MSELoss()
    
    # Training history
    history = {'loss': [], 'lr': []}
    best_loss = float('inf')
    patience_counter = 0
    
    model.train()
    for epoch in range(config.EPOCHS):
        epoch_loss = 0
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            
            optimizer.zero_grad()
            pred = model(bx)
            loss = criterion(pred, by)
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(loader)
        history['loss'].append(avg_loss)
        history['lr'].append(optimizer.param_groups[0]['lr'])
        
        scheduler.step(avg_loss)
        
        # Early stopping check
        if avg_loss < best_loss:
            best_loss = avg_loss
            patience_counter = 0
            best_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        # Logging
        if (epoch + 1) % 25 == 0:
            print(f"Epoch {epoch+1:3d}: Loss = {avg_loss:.6f}, LR = {optimizer.param_groups[0]['lr']:.2e}")
        
        # Early stopping
        if patience_counter >= config.PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Restore best model
    model.load_state_dict(best_state)
    print(f"\n✓ Training complete. Best loss: {best_loss:.6f}")
    
    return model, history

In [ ]:
# Train the model
model, history = train_delta_model(model, X_train, Y_train, config, device)

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['loss'], 'b-', linewidth=1)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss (MSE)')
ax1.set_title('Training Loss')
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

ax2.plot(history['lr'], 'r-', linewidth=1)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule')
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Recursive Delta Forecasting

In [ ]:
def recursive_forecast_delta(model, initial_context, n_steps, all_feat_norm, 
                             start_idx, delta_scaler, config, device):
    """
    Recursive forecasting using delta integration:
        y_{t+h} = y_t + predicted_delta
    
    Args:
        model: Trained LSTM
        initial_context: (SEQ_LEN, 15) normalized input sequence
        n_steps: Number of steps to forecast
        all_feat_norm: Full normalized dataset (for pH values)
        start_idx: Starting index in full dataset
        delta_scaler: Scaler for delta inverse transform
        config: Configuration object
        device: torch device
    
    Returns:
        predictions: (n_steps, 14) normalized state predictions
    """
    print("\n" + "="*60)
    print("[4/6] Recursive Delta Forecasting")
    print("="*60)
    
    model.eval()
    context = initial_context.copy()
    current_state_norm = context[-1, :config.N_STATES].copy()
    
    predictions = []
    horizon = config.PRED_HORIZON
    n_chains = (n_steps + horizon - 1) // horizon
    
    print(f"Forecasting {n_steps} steps ({n_chains} chains of {horizon})")
    
    with torch.no_grad():
        for chain in range(n_chains):
            # Predict delta
            ctx_tensor = torch.FloatTensor(context).unsqueeze(0).to(device)
            pred_delta_scaled = model(ctx_tensor).cpu().numpy()[0]
            
            # Inverse scale delta
            pred_delta = delta_scaler.inverse_transform(pred_delta_scaled.reshape(1, -1))[0]
            
            # Integrate: y_{t+h} = y_t + delta
            next_state_norm = current_state_norm + pred_delta
            
            # Store predictions for each step in horizon
            for step in range(horizon):
                if len(predictions) < n_steps:
                    predictions.append(next_state_norm.copy())
            
            # Update context for next chain
            # Use actual pH from dataset if available, otherwise repeat last
            future_idx = start_idx + (chain + 1) * horizon
            if future_idx < len(all_feat_norm):
                future_pH = all_feat_norm[future_idx, 14]
            else:
                future_pH = context[-1, 14]
            
            new_row = np.hstack([next_state_norm, future_pH])
            
            # Slide context window
            context = np.vstack([context[horizon:], np.tile(new_row, (horizon, 1))])
            current_state_norm = next_state_norm
    
    print(f"✓ Generated {len(predictions)} predictions")
    return np.array(predictions[:n_steps])

In [ ]:
# Run forecast from test set
start_idx = config.TRAIN_SIZE + 50  # Start in test region
initial_context = all_feat_norm[start_idx - config.SEQ_LEN : start_idx]

preds_norm = recursive_forecast_delta(
    model, initial_context, config.FORECAST_STEPS,
    all_feat_norm, start_idx, delta_scaler, config, device
)

## 11. Inverse Transform & Evaluation

In [ ]:
def inverse_transform_predictions(preds_norm, feat_scaler, config):
    """
    Inverse transform normalized predictions back to original scale.
    """
    print("\n" + "="*60)
    print("[5/6] Inverse Transform")
    print("="*60)
    
    # Pad with zeros for pH column (not predicted)
    preds_padded = np.hstack([preds_norm, np.zeros((len(preds_norm), 1))])
    
    # Inverse StandardScaler
    preds_orig = feat_scaler.inverse_transform(preds_padded)
    
    # Inverse log transform
    preds_final = preds_orig.copy()
    log_indices = [c for c in config.LOG_COLS if c < config.N_STATES]
    preds_final[:, log_indices] = np.expm1(preds_final[:, log_indices])
    
    # Ensure non-negative
    preds_final = np.maximum(preds_final, 0)
    
    print(f"✓ Inverse transform complete")
    return preds_final[:, :config.N_STATES]

In [ ]:
# Inverse transform
preds_final = inverse_transform_predictions(preds_norm, feat_scaler, config)

# Ground truth
gt_orig = data_full[start_idx : start_idx + config.FORECAST_STEPS, :config.N_STATES]
t_forecast = t_eval[start_idx : start_idx + config.FORECAST_STEPS]

In [ ]:
# Calculate errors
print("\n" + "="*60)
print("[6/6] Evaluation Results")
print("="*60)

state_names = ['nH2_g', 'nCO2_g', 'nCH4_g', 'nH2S_g', 'H2_aq', 'CO2_aq',
               'SO4', 'FeS', 'X', 'Acetate', 'HCO3', 'S_tot', 'Lag', 'Fe_pool']

print("\nRMSE per state variable:")
print("-" * 40)
for i, name in enumerate(state_names):
    rmse = np.sqrt(np.mean((preds_final[:, i] - gt_orig[:, i])**2))
    rel_error = rmse / (np.mean(np.abs(gt_orig[:, i])) + 1e-10) * 100
    print(f"  {name:12s}: RMSE = {rmse:.6f}, RelErr = {rel_error:.2f}%")

## 12. Visualization

In [ ]:
# Plot key variables
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

plot_vars = [
    (0, 'nH2_g (mmol)', 'Gas H2'),
    (1, 'nCO2_g (mmol)', 'Gas CO2'),
    (2, 'nCH4_g (mmol)', 'Gas CH4'),
    (3, 'nH2S_g (mmol)', 'Gas H2S'),
    (6, 'SO4 (mmol/L)', 'Sulfate'),
    (8, 'X (mmol/L)', 'Biomass')
]

for ax, (idx, ylabel, title) in zip(axes.flat, plot_vars):
    ax.plot(t_forecast, gt_orig[:, idx], 'b-', linewidth=2, label='Truth')
    ax.plot(t_forecast, preds_final[:, idx], 'r--', linewidth=2, label='Delta LSTM')
    ax.set_xlabel('Time (days)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Delta Learning Forecast ({config.FORECAST_STEPS} steps)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed H2 plot with error bands
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# H2 comparison
ax1.plot(t_forecast, gt_orig[:, 0], 'b-', linewidth=2, label='Ground Truth')
ax1.plot(t_forecast, preds_final[:, 0], 'r--', linewidth=2, label='Delta LSTM Prediction')
ax1.fill_between(t_forecast, gt_orig[:, 0], preds_final[:, 0], alpha=0.3, color='gray')
ax1.set_xlabel('Time (days)')
ax1.set_ylabel('nH2_g (mmol)')
ax1.set_title('H2 Gas: Delta Learning Forecast')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Absolute error over time
error = np.abs(preds_final[:, 0] - gt_orig[:, 0])
ax2.plot(t_forecast, error, 'k-', linewidth=1.5)
ax2.fill_between(t_forecast, 0, error, alpha=0.3)
ax2.set_xlabel('Time (days)')
ax2.set_ylabel('Absolute Error (mmol)')
ax2.set_title('H2 Gas: Prediction Error Over Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 13. Save Results

In [ ]:
# Save model and scalers
save_dir = '/content' if IN_COLAB else config.BASE_DIR

torch.save(model.state_dict(), os.path.join(save_dir, 'delta_lstm_model.pt'))
with open(os.path.join(save_dir, 'delta_scalers.pkl'), 'wb') as f:
    pickle.dump({'feat_scaler': feat_scaler, 'delta_scaler': delta_scaler}, f)

print(f"Model saved to: {save_dir}/delta_lstm_model.pt")
print(f"Scalers saved to: {save_dir}/delta_scalers.pkl")

In [ ]:
# Download results (Colab only)
if IN_COLAB:
    from google.colab import files
    files.download(os.path.join(save_dir, 'delta_lstm_model.pt'))
    files.download(os.path.join(save_dir, 'delta_scalers.pkl'))

## Summary

### Key Findings
- Delta learning predicts $\Delta y$ instead of absolute $y$
- Forces model to learn actual dynamics rather than identity mapping
- Recursive integration: $y_{t+h} = y_t + \hat{\Delta y}$

### Next Steps
1. Compare with absolute prediction baseline
2. Test on other rock types (Sandstone, Calcite, Gypsum)
3. Evaluate Physics-Informed Neural Network (PINN) approach